# FPL MaskablePPO Training

Train a MaskablePPO agent across multiple historical FPL seasons.

**Pipeline:**
1. Build multi-season training env (2016-17 to 2022-23)
2. Build eval env (2023-24) for model selection
3. Train MaskablePPO with action masking
4. Evaluate on holdout season (2024-25)
5. Compare against no-op and random baselines

In [1]:
from pathlib import Path
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

DATA_DIR = Path('../data/raw')
PREDICTOR_MODEL_DIR = Path('../models/point_predictor')  # set to None to skip
RUN_DIR = Path('../runs/fpl_ppo')
RUN_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SEASONS = [
    '2016-17', '2017-18', '2018-19', '2019-20',
    '2020-21', '2021-22', '2022-23',
]
EVAL_SEASON = '2023-24'
HOLDOUT_SEASON = '2024-25'

# Set to None to train without the LightGBM predictor
USE_PREDICTOR = PREDICTOR_MODEL_DIR if PREDICTOR_MODEL_DIR.exists() else None
print(f'Predictor: {USE_PREDICTOR or "disabled"}')

Predictor: ..\models\point_predictor


## 1. Hyperparameters

In [2]:
TOTAL_TIMESTEPS = 200_000  # ~5,263 episodes (~5h on CPU)
N_STEPS = 2048             # ~54 full episodes per rollout (episode = 38 steps)
BATCH_SIZE = 256           # 8 minibatches per epoch
N_EPOCHS = 10
LEARNING_RATE = 3e-4
GAMMA = 0.99               # GW38 discounted to 0.69x
ENT_COEF = 0.02            # mild exploration in large action space
NET_ARCH = [256, 256]
SEED = 42

EVAL_FREQ = 10_000         # evaluate every N timesteps
N_EVAL_EPISODES = 3
HOLDOUT_EPISODES = 5

print(f'Total timesteps:   {TOTAL_TIMESTEPS:,}')
print(f'Steps per rollout: {N_STEPS}')
print(f'Batch size:        {BATCH_SIZE}')
print(f'Approx episodes:   ~{TOTAL_TIMESTEPS // 38:,}')

Total timesteps:   200,000
Steps per rollout: 2048
Batch size:        256
Approx episodes:   ~5,263


## 2. Build Environments

In [3]:
from fpl_rl.training.multi_season_env import MultiSeasonFPLEnv
from fpl_rl.env.fpl_env import FPLEnv

PREDICTION_DATA_DIR = DATA_DIR.parent  # data/ (parent of data/raw/) for id_maps, understat, etc.

print(f'Building training env ({len(TRAIN_SEASONS)} seasons)...')
train_env = MultiSeasonFPLEnv(
    seasons=TRAIN_SEASONS,
    data_dir=DATA_DIR,
    predictor_model_dir=USE_PREDICTOR,
    prediction_data_dir=PREDICTION_DATA_DIR if USE_PREDICTOR else None,
    shuffle=False,
)

print(f'\nBuilding eval env ({EVAL_SEASON})...')
eval_kwargs = dict(season=EVAL_SEASON, data_dir=DATA_DIR)
if USE_PREDICTOR is not None:
    from fpl_rl.prediction.integration import PredictionIntegrator
    eval_kwargs['prediction_integrator'] = PredictionIntegrator.from_model(
        USE_PREDICTOR, PREDICTION_DATA_DIR, EVAL_SEASON,
    )
eval_env = FPLEnv(**eval_kwargs)

print(f'\nAction space:      {train_env.action_space}')
print(f'Observation space: {train_env.observation_space.shape}')
print('Done.')

Building training env (7 seasons)...


Caching predictions:   0%|          | 0/7 [00:00<?, ?season/s]

Seasons:   0%|          | 0/1 [00:00<?, ?season/s]

Understat players:   0%|          | 0/683 [00:00<?, ?player/s]

Seasons:   0%|          | 0/1 [00:00<?, ?season/s]

...

Understat players:   0%|          | 0/778 [00:00<?, ?player/s]


Building eval env (2023-24)...


Seasons:   0%|          | 0/1 [00:00<?, ?season/s]

Understat players:   0%|          | 0/865 [00:00<?, ?player/s]


Action space:      MultiDiscrete([ 3 15 50 15 50 15 15  6])
Observation space: (1363,)
Done.


## 3. Create Model & Callbacks

In [4]:
from sb3_contrib import MaskablePPO
from fpl_rl.training.callbacks import FPLEvalCallback, FPLEpisodeLogCallback

model = MaskablePPO(
    'MlpPolicy',
    train_env,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    learning_rate=LEARNING_RATE,
    gamma=GAMMA,
    ent_coef=ENT_COEF,
    policy_kwargs=dict(net_arch=NET_ARCH),
    tensorboard_log=str(RUN_DIR / 'tb_logs'),
    seed=SEED,
    verbose=0,
)

best_model_path = RUN_DIR / 'best_model'
best_model_path.mkdir(parents=True, exist_ok=True)

eval_cb = FPLEvalCallback(
    eval_env=eval_env,
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    best_model_save_path=best_model_path,
    verbose=1,
    show_progress=True,
)
episode_cb = FPLEpisodeLogCallback(verbose=0)

print(f'Model policy: {model.policy}')
n_params = sum(p.numel() for p in model.policy.parameters())
print(f'Parameters:   {n_params:,}')
print(f'Eval freq:    every {EVAL_FREQ:,} steps ({N_EVAL_EPISODES} episodes)')
print(f'Best model -> {best_model_path}')

Model policy: MaskableActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=1363, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=1363, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=256, out_features=169, bias=True)
  (value_net): Linear(in_features=256, out_features=1, bias=True)
)
Parameters:   873,642
Eval freq:    every 10,000 steps (3 episodes)
Best model -> ..\runs\fpl_pp

## 4. Train

SB3's `progress_bar=True` shows a tqdm bar over total timesteps.

In [5]:
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[eval_cb, episode_cb],
    progress_bar=True,
)

print(f'\nTraining complete.')
print(f'Best eval points: {eval_cb.best_mean_points:.1f}')
print(f'Training episodes: {episode_cb._episode_count}')

Output()

Eval episodes:   0%|          | 0/3 [00:00<?, ?ep/s]

KeyboardInterrupt: 

## 5. Save Final Model

In [ ]:
final_path = RUN_DIR / 'final_model'
final_path.mkdir(parents=True, exist_ok=True)
model.save(str(final_path / 'final_model'))
print(f'Final model saved to {final_path.resolve()}')
print(f'Files: {[f.name for f in final_path.iterdir()]}')

## 6. Holdout Evaluation (2024-25)

Run the trained agent, a no-op baseline, and a random-masked baseline on the holdout season.

In [ ]:
from tqdm.auto import tqdm

print(f'Building holdout env ({HOLDOUT_SEASON})...')
holdout_kwargs = dict(season=HOLDOUT_SEASON, data_dir=DATA_DIR)
if USE_PREDICTOR is not None:
    holdout_kwargs['prediction_integrator'] = PredictionIntegrator.from_model(
        USE_PREDICTOR, PREDICTION_DATA_DIR, HOLDOUT_SEASON,
    )
holdout_env = FPLEnv(**holdout_kwargs)
print('Done.')

In [ ]:
def run_agent(model, env, n_episodes, desc='Agent'):
    """Run trained agent for n episodes, return list of total points."""
    results = []
    for _ in tqdm(range(n_episodes), desc=desc, unit='ep'):
        obs, _ = env.reset()
        done = False
        pts = 0.0
        while not done:
            masks = env.action_masks()
            action, _ = model.predict(obs, deterministic=True, action_masks=masks)
            obs, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            pts = info.get('total_points', pts)
        results.append(pts)
    return results


def run_noop(env, n_episodes):
    """No-op baseline: zero action every GW."""
    results = []
    for _ in tqdm(range(n_episodes), desc='No-op', unit='ep'):
        obs, _ = env.reset()
        done = False
        pts = 0.0
        while not done:
            action = np.zeros(env.action_space.shape, dtype=int)
            obs, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            pts = info.get('total_points', pts)
        results.append(pts)
    return results


def run_random(env, n_episodes, seed=42):
    """Random masked baseline: sample valid actions."""
    rng = np.random.default_rng(seed)
    results = []
    for _ in tqdm(range(n_episodes), desc='Random', unit='ep'):
        obs, _ = env.reset()
        done = False
        pts = 0.0
        while not done:
            masks = env.action_masks()
            action = np.zeros(env.action_space.shape, dtype=int)
            offset = 0
            for dim_i, n_choices in enumerate(env.action_space.nvec):
                dim_mask = masks[offset:offset + n_choices]
                valid = np.where(dim_mask)[0]
                if len(valid) > 0:
                    action[dim_i] = rng.choice(valid)
                offset += n_choices
            obs, _, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            pts = info.get('total_points', pts)
        results.append(pts)
    return results

In [ ]:
agent_pts = run_agent(model, holdout_env, HOLDOUT_EPISODES, desc='PPO agent')
noop_pts = run_noop(holdout_env, HOLDOUT_EPISODES)
random_pts = run_random(holdout_env, HOLDOUT_EPISODES)

In [ ]:
def fmt(pts):
    return f'{np.mean(pts):7.1f} +/- {np.std(pts):5.1f}  (min={min(pts):.0f}, max={max(pts):.0f})'

print('=' * 65)
print(f'  Holdout evaluation: {HOLDOUT_SEASON} ({HOLDOUT_EPISODES} episodes)')
print('=' * 65)
print(f"  {'PPO Agent:':<20} {fmt(agent_pts)}")
print(f"  {'No-op baseline:':<20} {fmt(noop_pts)}")
print(f"  {'Random baseline:':<20} {fmt(random_pts)}")
print('=' * 65)

## 7. Cleanup

In [ ]:
train_env.close()
eval_env.close()
holdout_env.close()
print('Environments closed.')

---

## TensorBoard

```bash
tensorboard --logdir runs/fpl_ppo/tb_logs
```

Key metrics:
- `eval/mean_total_points` — held-out season performance
- `train/episode_total_points` — training curve
- `train/entropy_loss` — exploration level

## Loading a Saved Model

```python
from sb3_contrib import MaskablePPO
model = MaskablePPO.load('runs/fpl_ppo/best_model/best_model')
```